# UVERA · NB05 — AI-5 QR Shield

| | |
|---|---|
| **Accelerator** | None (CPU) |
| **Internet** | On |
| **Secret** | — |
| **Inputs to add** | — |
| **Expected runtime** | ~10–20 min |
| **How to run** | **Save Version → Save & Run All (Commit)** — runs in the background, no 20-minute idle timeout, 12-hour cap |

**What this AI does (say this to judges):** AI-5 looks at each merchant's weekly QR payments and spots shops that seem to be used as a cash machine (people pay round amounts and get cash back, skipping cash-out fees and limits). It combines a supervised model, which compares each shop with shops of the same type, area and size, with an unsupervised anomaly detector (70/30 rank fusion); the peer z-score alone is the transparent baseline. Analysts review the top 20 merchant-weeks each week. One disguise style (family D) is hidden from training to test whether it can catch a new trick. It never punishes anyone automatically: it asks for evidence and a person decides.

**Outputs** (in `/kaggle/working/outputs/NB05_outputs.zip`): `artifacts/ai5/` (model.txt, calibrator.json, conformal.json, thresholds.json, merchant_scores.parquet, zone_summary.parquet), `reports/metrics_ai5.json`, figures

All data is synthetic or open-licensed. No real customer data.

In [ ]:
# 1) Get the UVERA code from the public GitHub repo (nothing to upload by hand)
REPO = "https://github.com/ABRUBAB/takaresolve-ai-hackathon-2026.git"
REF = "main"   # replace with a commit hash to reproduce an exact run
import os, sys, subprocess
if not os.path.exists("/tmp/uvera"):
    subprocess.run(["git", "clone", "-q", REPO, "/tmp/uvera"], check=True)
subprocess.run(["git", "-C", "/tmp/uvera", "fetch", "-q", "origin", REF], check=False)
subprocess.run(["git", "-C", "/tmp/uvera", "checkout", "-q", REF], check=True)
COMMIT = subprocess.run(["git", "-C", "/tmp/uvera", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
os.environ["UVERA_ROOT"] = "/tmp/uvera"
sys.path.insert(0, "/tmp/uvera/ml")   # use Kaggle's own numpy/pandas/sklearn/lightgbm (no reinstall)
print("UVERA code at commit", COMMIT)
print(f'To reproduce this exact run later, set REF = "{COMMIT}" in this cell.')

In [ ]:
# 2) Reproducibility: seeds, versions, time budget, output folder
import json
from IPython.display import Image, display
from uvera_ml.common import env_info, set_seed, Budget, out_dir, zip_outputs, write_json
set_seed(42)
budget = Budget(minutes=90)
OUT = out_dir("NB05")
info = env_info(); info["uvera_commit"] = COMMIT
write_json(OUT / "reports" / "versions_nb05.json", info)
print(json.dumps(info, indent=1))

In [ ]:
# 3) Build the synthetic world. Same seed -> identical data on every Kaggle account (checked by hash).
from pathlib import Path
from uvera_ml.sim.world import load_or_generate
world = load_or_generate("/tmp/world_full", scale="full")
print(json.dumps(world.meta, indent=1)[:1500])
ref = Path("/tmp/uvera/reports/world_meta.json")
if ref.exists():
    expected = json.loads(ref.read_text(encoding="utf-8"))["meta"]["hashes"]["events"]
    print("Data identical to NB00 run:", expected == world.meta["hashes"]["events"])
budget.check("world")

## Train with merchant-grouped CV, test once
**StratifiedGroupKFold (5 folds, grouped by merchant) × 5 seeds** on families A, B, C; test merchants are never seen in training; family D only appears in the test.

In [ ]:
import pandas as pd
from uvera_ml.models.ai5 import run_ai5
summary = run_ai5(world, OUT, seeds=(42, 7, 1337, 2026, 99))
display(pd.DataFrame(summary["cv_summary_train"]))
print(json.dumps(summary["test"], indent=1, default=float)[:3000])
for f in ["ai5_unseen_family_D.png", "ai5_reliability.png"]:
    display(Image(str(OUT / "reports" / "figures" / f)))
budget.check("ai5")

In [ ]:
# Last step: package the results. Download NB05_outputs.zip from the Output tab.
budget.check("done")
zip_outputs(OUT)